# Model Klasifikasi Adidas, Converse, Nike (.h5)
**Nama:** Thariq Arkan | **NPM:** 2308108010065

Jalankan semua sel di Google Colab, lalu unggah `Hasil_roboflow.zip` saat diminta.

In [ ]:
#thrq
# Unggah Hasil_roboflow.zip (dataset yang sama dengan UTS)
import glob, os, zipfile
import numpy as np, pandas as pd, tensorflow as tf, yaml
from PIL import Image
from google.colab import files

unggah_thrq = files.upload()
with zipfile.ZipFile(list(unggah_thrq)[0]) as z_thrq:
    z_thrq.extractall('/content/dataset_thrq')
dir_data_thrq = os.path.dirname(glob.glob('/content/dataset_thrq/**/data.yaml', recursive=True)[0])
print('Folder dataset:', dir_data_thrq)

In [ ]:
#thrq
# Data klasifikasi: potongan tiap objek + gambar utuh (jika hanya berisi satu objek),
# karena pengguna aplikasi mengunggah gambar utuh, bukan potongan
import glob, os
import numpy as np, pandas as pd, tensorflow as tf, yaml
from PIL import Image

nama_kelas_thrq = yaml.safe_load(open(f'{dir_data_thrq}/data.yaml'))['names']

def siap_thrq(im_thrq):
    return np.array(im_thrq.resize((224, 224)), dtype='float32') / 255.0

def muat_data_thrq(subset_thrq):
    x_pot_thrq, y_pot_thrq, x_utuh_thrq, y_utuh_thrq = [], [], [], []
    for g_thrq in sorted(glob.glob(f'{dir_data_thrq}/{subset_thrq}/images/*')):
        im_thrq = Image.open(g_thrq).convert('RGB')
        lb_thrq = os.path.splitext(g_thrq.replace('/images/', '/labels/'))[0] + '.txt'
        baris_thrq = [l for l in open(lb_thrq).read().splitlines() if l.strip()]
        for l_thrq in baris_thrq:
            p_thrq = np.array(l_thrq.split(), float)
            if len(p_thrq) == 5:                     #thrq label kotak: x y w h
                cx, cy, w, h = p_thrq[1:]
                xy_thrq = np.array([[cx - w / 2, cy - h / 2], [cx + w / 2, cy + h / 2]])
            else:                                    #thrq label poligon
                xy_thrq = p_thrq[1:].reshape(-1, 2)
            kotak_thrq = (*(xy_thrq.min(0) * im_thrq.size), *(xy_thrq.max(0) * im_thrq.size))
            x_pot_thrq.append(siap_thrq(im_thrq.crop(kotak_thrq)))
            y_pot_thrq.append(int(p_thrq[0]))
        if len(baris_thrq) == 1:
            x_utuh_thrq.append(siap_thrq(im_thrq))
            y_utuh_thrq.append(int(baris_thrq[0].split()[0]))
    return tuple(np.array(a_thrq) for a_thrq in (x_pot_thrq, y_pot_thrq, x_utuh_thrq, y_utuh_thrq))

data_thrq = {s_thrq: muat_data_thrq(s_thrq) for s_thrq in ['train', 'valid', 'test']}
x_lat_thrq = np.concatenate([data_thrq['train'][0], data_thrq['train'][2]])
y_lat_thrq = np.concatenate([data_thrq['train'][1], data_thrq['train'][3]])
x_val_thrq = np.concatenate([data_thrq['valid'][0], data_thrq['valid'][2]])
y_val_thrq = np.concatenate([data_thrq['valid'][1], data_thrq['valid'][3]])
print('Data latih/valid:', len(x_lat_thrq), len(x_val_thrq))

In [ ]:
#thrq
# Transfer learning MobileNetV2: bobot ImageNet dibekukan, hanya lapisan akhir yang dilatih
tf.keras.utils.set_random_seed(42)
dasar_thrq = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
dasar_thrq.trainable = False

masuk_thrq = tf.keras.Input((224, 224, 3))
h_thrq = tf.keras.layers.RandomFlip('horizontal')(masuk_thrq)
h_thrq = tf.keras.layers.RandomRotation(0.05)(h_thrq)
h_thrq = tf.keras.layers.RandomZoom(0.1)(h_thrq)
h_thrq = tf.keras.layers.Rescaling(2.0, offset=-1.0)(h_thrq)   #thrq 0..1 menjadi -1..1
h_thrq = dasar_thrq(h_thrq, training=False)
h_thrq = tf.keras.layers.GlobalAveragePooling2D()(h_thrq)
h_thrq = tf.keras.layers.Dropout(0.2)(h_thrq)
keluar_thrq = tf.keras.layers.Dense(len(nama_kelas_thrq), activation='softmax')(h_thrq)
model_thrq = tf.keras.Model(masuk_thrq, keluar_thrq)

model_thrq.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_thrq.fit(x_lat_thrq, y_lat_thrq, validation_data=(x_val_thrq, y_val_thrq),
               epochs=100, batch_size=16,
               callbacks=[tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)])

In [ ]:
#thrq
# Fine-tuning: buka 30 lapisan terakhir MobileNetV2 dengan laju belajar kecil
dasar_thrq.trainable = True
for lapisan_thrq in dasar_thrq.layers[:-30]:
    lapisan_thrq.trainable = False
model_thrq.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_thrq.fit(x_lat_thrq, y_lat_thrq, validation_data=(x_val_thrq, y_val_thrq),
               epochs=100, batch_size=16,
               callbacks=[tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)])

In [ ]:
#thrq
# Evaluasi terpisah untuk potongan objek dan gambar utuh
for s_thrq in ['valid', 'test']:
    x_p, y_p, x_u, y_u = data_thrq[s_thrq]
    for nama_thrq, x_thrq, y_thrq in [('potongan', x_p, y_p), ('gambar utuh', x_u, y_u)]:
        pred_thrq = model_thrq.predict(x_thrq, verbose=0).argmax(1)
        print(f'{s_thrq.upper()} {nama_thrq}: akurasi {(pred_thrq == y_thrq).mean():.3f} '
              f'({(pred_thrq == y_thrq).sum()}/{len(y_thrq)})')
        print(pd.crosstab(pd.Series([nama_kelas_thrq[i] for i in y_thrq], name='Asli'),
                          pd.Series([nama_kelas_thrq[i] for i in pred_thrq], name='Prediksi')), '\n')

In [ ]:
#thrq
# Simpan model
model_thrq.save('classifier_model.h5', include_optimizer=False)
print('Ukuran berkas: %.2f MB' % (os.path.getsize('classifier_model.h5') / 1e6))

In [ ]:
#thrq
# Unduh ke komputer, lalu letakkan di folder model/ pada proyek Streamlit
files.download('classifier_model.h5')